# N4 code walkthrough, on the real data

Steps through the N4 code, mostly §19 (participants as the unit, shared trial
splits, local similarity, the overlap controls, Figure 5), on the **real
all-lPFC outputs**, printing the intermediate arrays at every step.

- The long-form explanation of each step is
  [`docs/n4_code_walkthrough.md`](../../docs/n4_code_walkthrough.md); section
  numbers here match it.
- The results and how to report them are §0 of
  [`docs/n4_continuous_anatomy.md`](../../docs/n4_continuous_anatomy.md).

**Run it on the DCC** from `dcc_scripts/stats/` (the same environment as the
other notebooks here). Cell 2 points at the run folders; edit the paths there if
yours differ. If a file is missing, the notebook says so. If the main anatomy
outputs are missing (for example on a laptop without the CSVs), it builds a
small **planted** data set with the same columns and runs the same code on it,
so every cell still works; the banner at the top of each result says which.

Permutation counts are lower than the pipeline's (`N_PERM` in cell 2) so cells
run in seconds; the point estimates do not depend on them. Set
`N_PERM = 10000` to reproduce the reported *p*-values.

## 0 · Setup

In [ ]:
%matplotlib inline
import inspect
import os
import sys
import tempfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import Image, display

warnings.simplefilter('ignore', FutureWarning)
pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 30)

# the repository root is two levels up from dcc_scripts/stats/
PROJECT_ROOT = Path(os.getcwd()).resolve()
while not (PROJECT_ROOT / 'src' / 'analysis').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.analysis.stats import stability_flexibility_anatomy as sfa
from src.analysis.stats import stability_flexibility_segregation as sfs


def show(fn, start=None, stop=None):
    """Print a function's source with its file line numbers. `start` and `stop`
    are text to match: print from the first line containing `start` through the
    next line containing `stop` (so the excerpt survives edits that move lines)."""
    lines, first = inspect.getsourcelines(fn)
    path = Path(inspect.getsourcefile(fn)).relative_to(PROJECT_ROOT)
    a = 0 if start is None else next(i for i, l in enumerate(lines) if start in l)
    b = len(lines) if stop is None else next(i for i, l in enumerate(lines) if i >= a and stop in l) + 1
    print(f'{path}')
    for i, l in enumerate(lines[a:b], start=first + a):
        print(f'{i:5d}  {l}', end='')


def banner(what):
    print(f'[{DATA} data] {what}')


print('project root:', PROJECT_ROOT)

### How the checks work

- **Questions**: answer with `ask('q1', 'a')`; it tells you whether you are right and why.
- **Predict first**: cells marked *Predict* ask you to write down what you expect before running the next cell.
- **Exercises**: compute something yourself in the cell marked `# YOUR CODE`, then run the `check(...)` cell under it. Solutions are folded under *Solution* in the markdown.

In [ ]:
import base64, json
QUIZ = {k: tuple(v) for k, v in json.loads(base64.b64decode('eyJxMSI6IFsiYSIsICJ5QiBpcyBMV1BTICh5KSBzY29yZWQgb24gaGFsZiBCIG9mIHRoYXQgc3BsaXQuIHggPSBMV1BDLCBteCA9IGNvbmdydWVuY3ksIG15ID0gc3dpdGNoOyBBL0IgYXJlIHRoZSB0d28gaGFsdmVzIG9mIG9uZSByYW5kb20gc3BsaXQuIl0sICJxMiI6IFsiYiIsICJXaXRoIHVuZXF1YWwgdHJpYWwgY291bnRzIHBlciBjZWxsICg3NSAlIGluY29uZ3J1ZW50IGJsb2NrcyBoYXZlIGZldyBjb25ncnVlbnQgdHJpYWxzKSwgYSB0cmlhbC13ZWlnaHRlZCBjb250cmFzdCBsZXRzIHRoZSBmcmVxdWVudCBjZWxscyBkb21pbmF0ZSBhbmQgbGVha3MgbWFpbiBlZmZlY3RzIGludG8gdGhlIGludGVyYWN0aW9uLiBFcXVhbCBjZWxsIHdlaWdodHMgbWFrZSBMV1BDIGFuZCB0aGUgY29uZ3J1ZW5jeSBtYWluIGVmZmVjdCBvcnRob2dvbmFsIGNvbnRyYXN0cyBvZiB0aGUgZm91ciBjZWxsIG1lYW5zLiJdLCAicTMiOiBbImMiLCAiRWFjaCBlbGVjdHJvZGUgZHJhd3MgaXRzIG93biByYW5kb20gaGFsZiwgc28gZWxlY3Ryb2RlIGoncyBoYWxmIEIgaXMgYW4gaW5kZXBlbmRlbnQgaGFsZiBvZiB0aGUgc2FtZSB0cmlhbHM6IGFib3V0IGhhbGYgb2YgZWxlY3Ryb2RlIGkncyBoYWxmLUEgdHJpYWxzIGxhbmQgaW4gaXQgKGEgcXVhcnRlciBvZiBhbGwgdHJpYWxzKS4gV2l0aCBhIHNoYXJlZCBzcGxpdCBpdCBpcyAwLiJdLCAicTQiOiBbImIiLCAiVHJpYWwtdG8tdHJpYWwgbm9pc2UgaW4gb25lIGhhbGYgZW50ZXJzIGV2ZXJ5IHNjb3JlIGNvbXB1dGVkIGZyb20gdGhhdCBoYWxmLiBJZiBMV1BDIGFuZCBMV1BTIGNhbWUgZnJvbSB0aGUgc2FtZSB0cmlhbHMsIHRoYXQgc2hhcmVkIG5vaXNlIGFsb25lIGNvdWxkIGNvcnJlbGF0ZSB0aGVtLiBPcHBvc2l0ZSBoYWx2ZXMgc2hhcmUgbm8gdHJpYWxzLCBzbyBpdCBjYW5ub3QuIl0sICJxNSI6IFsiYyIsICJDZW50cmluZyB3aXRoaW4gcGFydGljaXBhbnQgYW5kIHBlcm11dGluZyB3aXRoaW4gcGFydGljaXBhbnQgbWFrZSB0aGUgdGVzdCBhYm91dCBlbGVjdHJvZGVzIFdJVEhJTiBhIHBhcnRpY2lwYW50OiBhIHBhcnRpY2lwYW50IHdob3NlIGVsZWN0cm9kZXMgYXJlIGFsbCBzdHJvbmcgb24gYm90aCBlZmZlY3RzIGNhbm5vdCBjcmVhdGUgdGhlIGNvcnJlbGF0aW9uIGJ5IGl0c2VsZi4iXSwgInE2IjogWyJiIiwgIlRoZSB3ZWlnaHQgaXMgdGhlIHN1bSBvZiBzcXVhcmVkIChjbGVhbmVkKSBoZWlnaHRzLCBhYm91dCBuX2VsZWN0cm9kZXMgeCBzcHJlYWReMi4gQTogNDAgeCA1XjIgPSAxLDAwMC4gQjogOCB4IDQwXjIgPSAxMiw4MDAuIFNwcmVhZCBtYXR0ZXJzIGZhciBtb3JlIHRoYW4gZWxlY3Ryb2RlIGNvdW50LiJdLCAicTciOiBbImMiLCAiV2l0aCBldmVyeSBwcmVkaWN0b3IgY2VudHJlZCB3aXRoaW4gcGFydGljaXBhbnQsIHRoZSByYW5kb20gaW50ZXJjZXB0IG9ubHkgYWJzb3JicyBwYXJ0aWNpcGFudCBtZWFucy4gVGhlIHNsb3BlJ3Mgc3RhbmRhcmQgZXJyb3Igc3RpbGwgdHJlYXRzIGV2ZXJ5IGVsZWN0cm9kZSBhcyBpbmRlcGVuZGVudCwgc28gaXQgcmVwcm9kdWNlcyB0aGUgZWxlY3Ryb2RlLWxldmVsIHRlc3QuIE9ubHkgdGhlIHJhbmRvbSBTTE9QRSBsZXRzIHBhcnRpY2lwYW50cycgc2xvcGVzIGRpZmZlci4iXSwgInE4IjogWyJhIiwgIldlaWdodGVkIGJ5IGVsZWN0cm9kZSBjb3VudCB0aGUgb3ZlcmxhcCBpcyB0aGVyZTsgd2l0aCBldmVyeSBwYXJ0aWNpcGFudCBjb3VudGluZyBvbmNlIGl0IGlzIG5vdC4gU28gaXQgaXMgYSBwcm9wZXJ0eSBvZiB0aGUgZWxlY3Ryb2RlIHBvcHVsYXRpb24sIGNhcnJpZWQgYnkgZWxlY3Ryb2RlLXJpY2ggcGFydGljaXBhbnRzLCBub3Qgc29tZXRoaW5nIGV2ZXJ5IHBhcnRpY2lwYW50IHNob3dzLiJdLCAicTkiOiBbImQiLCAiQ1tpLCBpXSBwYWlycyBlbGVjdHJvZGUgaSdzIGhhbGYgQSB3aXRoIGl0cyBvd24gaGFsZiBCOiBpdHMgc3BsaXQtaGFsZiByZWxpYWJpbGl0eS4gVGhlIG9mZi1kaWFnb25hbCBDW2ksIGpdIGlzIGhvdyBtdWNoIGVsZWN0cm9kZSBqJ3Mgc2NvcmUgcHJlZGljdHMgZWxlY3Ryb2RlIGkncyBhY3Jvc3MgaW5kZXBlbmRlbnQgdHJpYWxzLiJdLCAicTEwIjogWyJiIiwgIkEgbnVsbCBmb3IgdGhlIGJhbGFuY2UgbWVhbnMgJ2ludGVybWl4ZWQnIG9ubHkgaWYgdGhlIGJhbGFuY2UgaXRzZWxmIGlzIHJlbGlhYmxlIGFuZCB0aGUgc2luZ2xlIHNjb3JlcyBzaG93IHRoZSBhbmFseXNpcyBjYW4gc2VlIGxvY2FsIHN0cnVjdHVyZS4gT24gdGhlIHJlYWwgZGF0YSB0aGUgYmFsYW5jZSdzIHJlbGlhYmlsaXR5IGlzIGFib3V0IDA6IG5vdGhpbmcgcmVsaWFibGUgaXMgbGVmdCB0byBiZSBwYXRjaHkgb3IgaW50ZXJtaXhlZC4iXSwgInExMSI6IFsiYSIsICJFc3RpbWF0aW9uIG5vaXNlIGluIG9uZSBkYXRhIHNldCBpcyBpdHNlbGYgc3BhdGlhbGx5IHNtb290aCAobmVpZ2hib3VycyBzaGFyZSBpdCksIHNvIHBhaXJzIGFyZSBub3QgZXhjaGFuZ2VhYmxlOyBhIHBhaXItbGV2ZWwgbnVsbCB3YXMgdG9vIGxpYmVyYWwgKDEzICUgZmFsc2UgcG9zaXRpdmVzIG9uIHNpbXVsYXRlZCBkYXRhKS4gRmxpcHBpbmcgd2hvbGUgcGFydGljaXBhbnRzJyBleGNlc3NlcyB0cmVhdHMgcGFydGljaXBhbnRzIGFzIHRoZSB1bml0cyAoMi41ICUpLiJdLCAicTEyIjogWyJjIiwgIlBvaW50cyBhcmUgKExXUEMsIExXUFMpIGluIFNEIHVuaXRzLiBBYm92ZSB0aGUgaWRlbnRpdHkgbGluZSwgTFdQUyA+IExXUEM6IHRoZSBlbGVjdHJvZGUgbGVhbnMgTFdQUy4gTFdQQyAtIExXUFMgaXMgZWFjaCBwb2ludCdzIHNpZ25lZCBkaXN0YW5jZSBiZWxvdyB0aGUgbGluZS4iXSwgInExMyI6IFsiZCIsICJUaGUgYmFzZSBlZmZlY3RzIGFyZSBhbHNvIHRoZSBiZXN0IGF2YWlsYWJsZSBwcm94eSBmb3IgYW4gZWxlY3Ryb2RlJ3Mgc2lnbmFsLXRvLW5vaXNlLCBzbyB0aGlzIHJvdyBhbG9uZSBjYW5ub3QgdGVsbCAnZWFjaCBhZGFwdGF0aW9uIHNjYWxlcyB3aXRoIHRoZSBlZmZlY3QgaXQgcmVndWxhdGVzJyBmcm9tICdib3RoIHNjYWxlIHdpdGggcmVzaWR1YWwgc2lnbmFsLXRvLW5vaXNlJy4gVGhlIHByb2Nlc3Mtc3BlY2lmaWMgbWF0Y2hlZCA+IGNyb3NzZWQgcGF0dGVybiBhcmd1ZXMgYWdhaW5zdCBwdXJlIHNpZ25hbC10by1ub2lzZS4iXSwgInExNCI6IFsiYiIsICJ4QSBpcyBjbGVhbmVkIHdpdGggaGFsZi1BIGJhc2UgZWZmZWN0cyBhbmQgeUIgd2l0aCBoYWxmLUIgYmFzZSBlZmZlY3RzLCBzbyB0aGUgY29ycmVsYXRlZCBwYWlyIHN0aWxsIGhhcyBoYWxmLUEgdHJpYWxzIG9uIG9uZSBzaWRlIGFuZCBoYWxmLUIgdHJpYWxzIG9uIHRoZSBvdGhlci4gQ2xlYW5pbmcgeEEgd2l0aCBoYWxmLUIgZWZmZWN0cyB3b3VsZCBjYXJyeSBoYWxmLUIgdHJpYWwgbm9pc2UgaW50byBpdC4iXX0=')).items()}  # the answer key, encoded

def ask(qid, choice):
    """Answer a multiple-choice question: ask('q6', 'b')."""
    right, why = QUIZ[qid]
    ok = str(choice).strip().lower() == right
    print(('Correct. ' if ok else f'Not quite (the answer is {right}). ') + why)


def check(name, value, expected, tol=1e-6, hint=''):
    """Compare an exercise answer with the pipeline's value."""
    try:
        good = (abs(float(value) - float(expected)) <= tol * max(1.0, abs(float(expected))))
    except (TypeError, ValueError):
        good = value == expected
    print(f'{name}: ' + ('correct' if good else f'not yet (got {value!r}, expected {expected!r}). {hint}'))

## 1 · Where the data are

The defaults are the folders that produced the current results (§0.2 and §19.8.1
of the N4 doc), relative to the repository, which is how they sit on the DCC
under `/hpc/home/jz421/coganlab/jz421/GlobalLocal/`.

| Variable | Folder | Used for |
|---|---|---|
| `ANATOMY_DIR` | the all-lPFC anatomy run's `continuous/` | `scores_with_anatomy.csv`, `per_split.csv` (everything) |
| `SEG_DIR` | the all-lPFC `_main_effects` segregation run | panel b's *r* (`correlation.json`) |
| `SHARED_PER_SPLIT` | `section19/per_split_shared.csv` from the 2026-10-05 raw run | local similarity (§4.3) without rescoring |
| `LONG_DF` | the raw scatter-only run's `long_df.csv` | scoring one electrode by hand (§3.1), shared halves, and rescoring if `SHARED_PER_SPLIT` is missing |
| `RT_COUPLING` | the RT-adjusted scatter-only run's `rt_adjustment_slopes.csv` | the RT row of the overlap controls (§4.5) |

In [ ]:
EPOCHS = ('Stimulus_-1.0to1.5sec_decFactor_8_outliers_10_drop_and_nan_thresh_perc_5.0_'
          '70.0-150.0_Hz_padLength_1.5s_filterbank_hilbert_stat_func_ttest_zmax_20')
RESULTS = PROJECT_ROOT / 'dcc_scripts' / 'stats' / 'results' / EPOCHS
SEG_ROOT = RESULTS / 'segregation_results'

ANATOMY_DIR = RESULTS / 'anatomy_a1_lpfc_window_0.0to1.5s_sig' / 'continuous'
SEG_DIR = SEG_ROOT / 'window_0.0to1.5s_all_lpfc_proportion_cohens_d_fdr_bh_main_effects'
SHARED_PER_SPLIT = ANATOMY_DIR / 'section19' / 'per_split_shared.csv'
LONG_DF = SEG_ROOT / 'window_0.0to1.5s_all_lpfc_proportion_cohens_d_fdr_bh_scatter_only_splits0' / 'long_df.csv'
RT_COUPLING = (SEG_ROOT / 'window_0.0to1.5s_all_lpfc_proportion_cohens_d_fdr_bh_rt_adjusted_scatter_only_splits200'
               / 'rt_adjustment_slopes.csv')

N_PERM = 2000          # 10000 reproduces the reported p-values
N_BOOT = 1000          # 2000 in the pipeline
SHARED_N_SPLITS = 200  # only used if the shared table has to be rescored from LONG_DF
FORCE_PLANTED = False  # True runs the planted data even when the real files exist
DRAW_BRAIN = False     # True draws Figure 5's tertile brain (needs the recon files and a display, e.g. xvfb)

for p in (ANATOMY_DIR / 'scores_with_anatomy.csv', ANATOMY_DIR / 'per_split.csv',
          SEG_DIR / 'correlation.json', SHARED_PER_SPLIT, LONG_DF, RT_COUPLING):
    print(f"{'found  ' if p.exists() else 'MISSING'}  {p}")

### Load, or plant

The planted fallback builds a **trial-level** table (contacts on shafts, trial
noise shared between neighbouring contacts, a base-effect factor that both
adaptations scale with, and a height gradient in LWPC), then runs the real
scoring code on it. So the fallback exercises the same functions, including
the shared split. Its numbers are made up: the world is built to resemble the
real story (an overlap carried by the base effects, a height gradient), but
only the real data say anything about lPFC.

In [ ]:
def planted_long_table(n_subj=8, n_shafts=3, per_shaft=6, trials_per_block=80, seed=0):
    """Trial-level table with the long table's columns, and the contacts' MNI coordinates."""
    from src.analysis.stats.stability_flexibility_brain_behavior import _BLOCK_PROPORTION_MAP
    rng = np.random.default_rng(seed)
    frames, coords = [], {}
    for s in range(n_subj):
        subj = f'P{s:02d}'
        P = []
        for c in rng.uniform([-50, 10, -5], [-25, 50, 60], size=(n_shafts, 3)):
            u = rng.normal(size=3)
            P += [c + k * 3.5 * u / np.linalg.norm(u) for k in range(per_shaft)]
        P = np.asarray(P)
        D = np.linalg.norm(P[:, None] - P[None], axis=-1)
        L = np.linalg.cholesky(np.exp(-(D / 5.0) ** 2) + 1e-6 * np.eye(len(P)))
        cols = dict(congruency=[], switchType=[], incongruent_proportion=[], switch_proportion=[])
        for props in _BLOCK_PROPORTION_MAP.values():
            n_i = round(trials_per_block * props['incongruent_proportion'] / 100)
            n_s = round(trials_per_block * props['switch_proportion'] / 100)
            cols['congruency'] += list(rng.permutation(['i'] * n_i + ['c'] * (trials_per_block - n_i)))
            cols['switchType'] += list(rng.permutation(['s'] * n_s + ['r'] * (trials_per_block - n_s)))
            cols['incongruent_proportion'] += [props['incongruent_proportion']] * trials_per_block
            cols['switch_proportion'] += [props['switch_proportion']] * trials_per_block
        n_t = len(cols['congruency'])
        inc = np.array(cols['congruency']) == 'i'
        sw = np.array(cols['switchType']) == 's'
        hc = np.where(np.array(cols['incongruent_proportion']) == 25.0, 0.5, -0.5)
        hs = np.where(np.array(cols['switch_proportion']) == 25.0, 0.5, -0.5)
        base = rng.normal(size=len(P))                         # shared base-effect factor
        cong = 0.3 + 0.25 * (base + 0.5 * rng.normal(size=len(P)))
        swe = 0.3 + 0.25 * (base + 0.5 * rng.normal(size=len(P)))
        lwpc = 0.15 + 0.8 * (cong - 0.3) - 0.012 * (P[:, 2] - 25) + 0.1 * rng.normal(size=len(P))
        lwps = 0.15 + 0.8 * (swe - 0.3) + 0.1 * rng.normal(size=len(P))
        gain = rng.uniform(0.5, 2.5, size=len(P))              # overall response level (drives resp)
        noise = 0.6 * (L @ rng.normal(size=(len(P), n_t)))     # neighbours share trial noise
        for e in range(len(P)):
            el = f'{subj}-L{e // per_shaft}{e % per_shaft + 1}'
            coords[el] = tuple(P[e])
            hg = gain[e] + inc * (cong[e] + hc * lwpc[e]) + sw * (swe[e] + hs * lwps[e]) + noise[e]
            frames.append(pd.DataFrame(dict(subject=subj, electrode=el, trial=np.arange(n_t),
                                            hg=hg, rt=np.nan, **cols)))
    return pd.concat(frames, ignore_index=True), coords


have_real = (ANATOMY_DIR / 'scores_with_anatomy.csv').exists() and (ANATOMY_DIR / 'per_split.csv').exists()
DATA = 'real' if have_real and not FORCE_PLANTED else 'PLANTED'

if DATA == 'real':
    scores = pd.read_csv(ANATOMY_DIR / 'scores_with_anatomy.csv')
    per_split = pd.read_csv(ANATOMY_DIR / 'per_split.csv')
    long_df = pd.read_csv(LONG_DF) if LONG_DF.exists() else None
    per_split_shared = pd.read_csv(SHARED_PER_SPLIT) if SHARED_PER_SPLIT.exists() else None
    rt = pd.read_csv(RT_COUPLING) if RT_COUPLING.exists() else None
    seg_dir = str(SEG_DIR) if SEG_DIR.exists() else None
else:
    print('Real outputs not found: building the planted data set (about a minute) ...')
    long_df, _coords = planted_long_table()
    per_split = sfs.compute_sensitivities_per_split(long_df, n_splits=24, seed=0,
                                                    contrast_mode='proportion', main_effects=True)
    scores = sfa.attach_scores(sfs.add_responsiveness(sfs.average_over_splits(per_split), long_df),
                               {}, electrodes_to_coords=_coords)
    per_split_shared, rt, seg_dir = None, None, None

banner(f"{scores['electrode'].nunique()} electrodes, {scores['subject'].nunique()} participants; "
       f"per-split table {per_split.shape[0]:,} rows, {per_split['split'].nunique()} splits")
print('long table:', 'none' if long_df is None else f"{len(long_df):,} rows, "
      f"trial column {'present' if 'trial' in long_df else 'MISSING'}")
print('shared per-split table:', 'none (will rescore if the long table has trial ids)'
      if per_split_shared is None else f"{per_split_shared['electrode'].nunique()} electrodes, "
      f"{per_split_shared['split'].nunique()} splits, scheme {set(per_split_shared.get('split_scheme', ['?']))}")
print('RT coupling:', 'none' if rt is None else f"{rt['electrode'].nunique()} electrodes")

### The real numbers already in the repo

The CSVs are not in git, but the summaries of the real runs are. This prints
them, so the real numbers are on screen even if this session is running on the
planted fallback.

In [ ]:
committed = {
    '§19, first run (all lPFC; its local-similarity block is the unusable per-electrode version)':
        ANATOMY_DIR / 'section19' / 'summary_section19.txt',
    'segregation, all lPFC, raw': SEG_DIR / 'summary.txt',
    'segregation, all lPFC, RT-adjusted':
        SEG_ROOT / 'window_0.0to1.5s_all_lpfc_proportion_cohens_d_fdr_bh_main_effects_rt_adjusted' / 'summary.txt',
}
for title, p in committed.items():
    print('=' * 100, f'\n{title}\n{p.relative_to(PROJECT_ROOT) if p.is_relative_to(PROJECT_ROOT) else p}')
    if not p.exists():
        print('  (not found)')
        continue
    text = p.read_text().splitlines()
    keep = text if 'section19' in str(p) else [l for l in text if l.strip().startswith(('CONTINUOUS', 'n_electrodes', 'ceiling', 'MAIN EFFECTS', 'rt_adjust_hg', 'n_splits'))]
    print('\n'.join(keep))

## 2 · The tables

`scores` is one row per electrode (`scores_with_anatomy.csv`), `per_split` one
row per electrode × split. x = LWPC, y = LWPS, mx = congruency, my = switch;
A and B are the two halves of a split. Check the electrode count matches the
run you expect (398 for all lPFC, 171 for the subset).

In [ ]:
banner('scores_with_anatomy.csv')
cols = [c for c in ('subject', 'electrode', 'lwpc_score', 'lwps_score', 'cong_score', 'switch_score',
                    'lwpc_s', 'lwps_s', 'delta', 'dm', 'resp', 'anat', 'mni_x', 'mni_y', 'mni_z')
        if c in scores]
display(scores[cols].head())
print('\nelectrodes per participant:')
print(scores.groupby('subject').size().sort_values(ascending=False).to_string())

In [ ]:
banner('per_split.csv (first electrode, first splits)')
display(per_split.head(6))
print('pooled scale factors (score / scaled score):',
      {k: round(float(np.nanmedian(scores[f'{k}_score'] / scores[f'{k}_s'])), 4)
       for k in ('lwpc', 'lwps', 'cong', 'switch') if f'{k}_s' in scores})

**Check yourself, q1.** In `per_split.csv`, what is `yB`?

- (a) LWPS scored on half B of that split
- (b) the switch main effect on half B
- (c) LWPS averaged over all splits
- (d) LWPC on half B

In [ ]:
ask('q1', '?')   # replace ? with your letter

## 3 · Building blocks

### 3.1 One electrode's score, by hand

`_interaction_cohens_d` combines the four cell means with fixed weights and
divides by the pooled within-cell SD. Below: LWPC for one electrode from all its
trials, by hand and by the pipeline's function. Needs the long table.

In [ ]:
show(sfs._interaction_cohens_d)
print()
print('W_INTERACTION =', sfs.W_INTERACTION)
print('W_MAIN        =', sfs.W_MAIN)

In [ ]:
if long_df is None:
    print('No long table: skipping (set LONG_DF in cell 2).')
else:
    el = long_df['electrode'].iloc[0]
    t = long_df[long_df['electrode'] == el].dropna(subset=['hg'])
    low = t['incongruent_proportion'] == t['incongruent_proportion'].min()     # 25 % incongruent
    cells = {('I', '25%'): t[(t.congruency == 'i') & low]['hg'],
             ('C', '25%'): t[(t.congruency == 'c') & low]['hg'],
             ('I', '75%'): t[(t.congruency == 'i') & ~low]['hg'],
             ('C', '75%'): t[(t.congruency == 'c') & ~low]['hg']}
    means = {k: v.mean() for k, v in cells.items()}
    dod = (means[('I', '25%')] - means[('C', '25%')]) - (means[('I', '75%')] - means[('C', '75%')])
    sp = np.sqrt(sum((len(v) - 1) * v.var(ddof=1) for v in cells.values())
                 / sum(len(v) - 1 for v in cells.values()))
    print(f'{el}: cell n =', {k: len(v) for k, v in cells.items()})
    print(f'by hand:  LWPC = d-o-d / pooled SD = {dod:.4f} / {sp:.4f} = {dod / sp:+.4f}')
    contrasts = sfs.finalize_contrasts(t, sfs.resolve_contrasts('proportion'))
    work = sfs._canonical_labels(t, contrasts)
    print(f"pipeline: {sfs._effect_for(work, 'stability', '_slab', contrasts, 'cohens_d', 0.05):+.4f}")

**Check yourself, q2.** Why does the score weight the four cell means equally instead of pooling trials?

- (a) It is faster to compute
- (b) So frequent cells cannot dominate, and main effects cannot leak into the interaction
- (c) Cohen's d is only defined for equal weights
- (d) So that LWPC comes out positive

In [ ]:
ask('q2', '?')   # replace ? with your letter

### 3.1b Per-electrode vs shared trial halves

With the default split each electrode draws its own halves, so electrode *i*'s
half A shares about half its trials with electrode *j*'s half B. With
`shared_split=True` every electrode of a participant uses the same halves, so
the overlap is zero. This is why local similarity and within-participant
reliabilities need the shared split (walkthrough §3.1).

*Predict:* with per-electrode splits, what fraction of electrode 1's half-A
trials will also be in electrode 2's half B? Write your guess down, then run.

In [ ]:
if long_df is None or 'trial' not in long_df:
    print('Needs a long table with a trial column.')
else:
    subj = long_df['subject'].iloc[0]
    sub = long_df[long_df['subject'] == subj]
    contrasts = sfs.finalize_contrasts(sub, sfs.resolve_contrasts('proportion'))
    work = sfs._canonical_labels(sub, contrasts)
    strata = sfs._strata_columns(contrasts)
    rng = np.random.default_rng(0)
    e1, e2 = sorted(work['electrode'].unique())[:2]

    def trials(e, idx):
        return set(work.loc[idx, 'trial'])

    w1 = work[work['electrode'] == e1]
    w2 = work[work['electrode'] == e2]
    a1, _ = sfs._stratified_half_split(w1, rng, strata_cols=strata)
    _, b2 = sfs._stratified_half_split(w2, rng, strata_cols=strata)
    shared_frac = len(trials(e1, a1) & trials(e2, b2)) / len(trials(e1, a1))
    print(f'per-electrode split: {shared_frac:.0%} of {e1} half A trials are in {e2} half B')

    halves = sfs._shared_trial_halves(work, strata, 5, rng)[subj]   # trial x split, 0 = A, 1 = B
    A = set(halves.index[halves[0] == 0]); B = set(halves.index[halves[0] == 1])
    print(f'shared split:        {len(A & B)} trials in both halves '
          f'({len(A)} in A, {len(B)} in B, the same for every electrode of {subj})')

**Check yourself, q3.** Per-electrode splits: about what fraction of electrode *i*'s half-A trials are also in electrode *j*'s half B?

- (a) 0 %
- (b) about 25 %
- (c) about 50 %
- (d) 100 %

In [ ]:
ask('q3', '?')   # replace ? with your letter

### 3.2–3.3 The overlap test, unpacked

`_residualised_split_matrices` gives one (splits × electrodes) array per score,
regressed on responsiveness and centred within participant.
`split_resolved_corr` turns each split's vector into a unit vector (after
ranking), so a correlation is a dot product, and averages LWPC-half-A ×
LWPS-half-B and the reverse over splits. Below, the same number by hand.

In [ ]:
show(sfs.split_resolved_corr, 'elecs, subj, groups, splits, mats, n_elec_in', 'p = float((np.sum')

In [ ]:
resp = scores.drop_duplicates('electrode').set_index('electrode')['resp']
elecs, subj, groups, splits, mats, n_in = sfs._residualised_split_matrices(per_split, resp, min_elec=3)
banner(f'{len(elecs)} electrodes in {len(groups)} participants (of {n_in}), '
       f'{len(splits)} splits; each matrix is {mats["xA"].shape}')

U = {k: np.vstack([sfs._unit_vector(mats[k][s], 'spearman') for s in range(len(splits))]) for k in mats}
M = 0.5 * (U['xA'].T @ U['yB'] + U['xB'].T @ U['yA']) / len(splits)
print(f'by hand: overlap r = trace(M) = {np.trace(M):+.4f}; '
      f'LWPC reliability = {(U["xA"] * U["xB"]).sum(1).mean():+.4f}, '
      f'LWPS reliability = {(U["yA"] * U["yB"]).sum(1).mean():+.4f}')

res = sfs.split_resolved_corr(per_split, resp, n_perm=N_PERM)
print(f"pipeline: r = {res['corr']:+.4f}, p = {res['p']:.4g} ({N_PERM} permutations), "
      f"reliabilities {res['reliability_x']:+.4f} / {res['reliability_y']:+.4f}")

**Exercise E1.** Compute LWPC's within-participant split-half reliability
yourself from `U`: in each split, the dot product of the half-A and half-B unit
vectors; then the mean over splits. Store it in `my_rel_x`.

<details><summary>Solution</summary>

`my_rel_x = (U['xA'] * U['xB']).sum(axis=1).mean()`

Each row of `U` is one split's centred, unit-length vector, so the row-wise dot
product is that split's correlation.
</details>

In [ ]:
my_rel_x = None   # YOUR CODE

In [ ]:
check('E1', my_rel_x, res['reliability_x'], hint="Row-wise product of U['xA'] and U['xB'], summed per row, then averaged.")

**Check yourself, q4.** Why does the overlap test pair LWPC from half A with LWPS from half B, not both from the same half?

- (a) To double the number of electrodes
- (b) Trial noise shared within a half could correlate the two scores by itself
- (c) Because LWPS is only defined on half B
- (d) To make the test one-sided

In [ ]:
ask('q4', '?')   # replace ? with your letter

**Check yourself, q5.** Why does the null permute LWPS **within** participant (the same permutation in every split)?

- (a) To save time
- (b) Because participants have different numbers of splits
- (c) So the test is about electrodes within a participant: participant-wide differences cannot create the correlation
- (d) So that the p-value is always below 0.05

In [ ]:
ask('q5', '?')   # replace ? with your letter

The diagonal of `M` is each electrode's own cross-half LWPC × LWPS product;
its off-diagonal entries are what the permutation null samples from. A quick
look at which electrodes contribute most:

In [ ]:
contrib = pd.DataFrame({'electrode': elecs, 'subject': subj, 'contribution': np.diag(M)})
print('participants by summed contribution to r:')
print(contrib.groupby('subject')['contribution'].agg(['sum', 'size']).sort_values('sum').to_string())

### 3.4 The coordinate test

`delta ~ y + z + x + resp + participant`, with the swap null (random sign flips
of delta). The *z* slope is the gradient everything in §4.1 decomposes.

In [ ]:
coord = sfa.relative_score_coordinate_test(scores, n_perm=N_PERM)
banner('coordinate test, all electrodes')
print(f"block F = {coord['all']['observed_stat']:.3f}, p = {coord['all']['p']:.4g}")
display(coord['all']['slopes'])

## 4 · §19, part by part

### 4.1 Section 1: the height slope with participants as the unit

The pooled slope is a weighted average of the participants' own slopes, each
weighted by its electrodes' spread in height (after the same covariates).
First the identity, by hand; then the function.

*Predict:* will the weighted mean of the participants' slopes equal the
coordinate test's slope exactly, or only approximately?

In [ ]:
show(sfa._partial_axis_residuals)

In [ ]:
d = scores.dropna(subset=['delta', 'mni_y', 'mni_z', 'mni_x']).reset_index(drop=True)
v, a = sfa._partial_axis_residuals(d, 'delta', 'mni_z', ('mni_y', 'mni_z', 'mni_x'), ('resp',))
rows = []
for s, g in d.groupby('subject').groups.items():
    w = (a[g] ** 2).sum()
    rows.append(dict(subject=s, n=len(g), weight=w, slope=(a[g] * v[g]).sum() / w if w > 1e-9 else np.nan))
by_hand = pd.DataFrame(rows).dropna()
pooled = (a * v).sum() / (a * a).sum()
banner(f'pooled slope {pooled:+.5f}/mm; weighted mean of participant slopes '
       f'{(by_hand.weight * by_hand.slope).sum() / by_hand.weight.sum():+.5f}/mm; '
       f"coordinate test {coord['all']['slopes'].set_index('axis').loc['mni_z', 'slope_per_mm']:+.5f}/mm")

In [ ]:
part = sfa.coordinate_slope_by_participant(scores, n_perm=N_PERM, n_boot=N_BOOT)
banner('coordinate_slope_by_participant')
print('\n'.join(sfa.participant_slope_lines(part)))
display(part['per_participant'].sort_values('weight', ascending=False))

In [ ]:
per = part['per_participant'].dropna(subset=['slope_per_mm'])
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.axhline(0, color='0.8', lw=0.8)
ax.axhline(part['pooled_slope'], color='C3', lw=1, label=f"weighted mean {part['pooled_slope']:+.4f}")
ax.scatter(per['spread_mm'], per['slope_per_mm'], s=per['n_electrodes'] * 4, alpha=0.6,
           c=np.where(per['in_unweighted_test'], 'C0', '0.6'))
for r in per.itertuples():
    ax.annotate(r.subject, (r.spread_mm, r.slope_per_mm), fontsize=6, alpha=0.7)
ax.set(xlabel='spread along z after covariates (mm)', ylabel='participant slope (SD/mm)',
       title=f'[{DATA}] each participant\'s own height slope (size = electrodes)')
ax.legend(frameon=False, fontsize=8)
display(fig); plt.close(fig)

In [ ]:
loso = sfa.coordinate_slope_loso(scores, n_perm=max(500, N_PERM // 4))
banner('leave one participant out (first row = all)')
display(loso)

**Exercise E2.** Which participant carries the most weight in the pooled
slope? Store its id in `my_heaviest` (use `part['per_participant']`). Then look
at its `spread_mm` and `n_electrodes`: is it many electrodes or a wide spread?

<details><summary>Solution</summary>

`my_heaviest = part['per_participant'].sort_values('weight').iloc[-1]['subject']`
</details>

In [ ]:
my_heaviest = None   # YOUR CODE

In [ ]:
check('E2', my_heaviest, part['per_participant'].sort_values('weight').iloc[-1]['subject'])

**Check yourself, q6.** Participant A has 40 electrodes spread over 5 mm in height; B has 8 electrodes spread over 40 mm. Who weighs more in the pooled slope?

- (a) A, because it has five times the electrodes
- (b) B, because weight grows with the square of the spread
- (c) They weigh the same
- (d) Neither: every participant weighs the same

In [ ]:
ask('q6', '?')   # replace ? with your letter

**Check yourself, q7.** Why is the random-**intercept** model's p = 0.005 not a participant-level test?

- (a) It failed to converge
- (b) It uses Spearman instead of Pearson
- (c) With predictors centred within participant it reproduces the electrode-level slope and its precision; only a random slope lets participants' slopes differ
- (d) Because it drops participants with fewer than three electrodes

In [ ]:
ask('q7', '?')   # replace ? with your letter

### 4.2 Section 2: the overlap r with participants as the unit

The same residualised values as §3.3, correlated within each participant, then
combined in Fisher *z* with weights *n* − 3.

In [ ]:
pc = sfs.participant_split_corr(per_split, resp, n_perm=N_PERM, n_boot=N_BOOT)
banner('participant_split_corr')
print('\n'.join(sfa.participant_corr_lines(pc)))
pp = pc['per_participant'].sort_values('n_electrodes')
display(pp)

fig, ax = plt.subplots(figsize=(6, 3.4))
ax.axhline(0, color='0.8', lw=0.8)
ax.axhline(pc['corr_weighted'], color='C3', lw=1, label=f"weighted r {pc['corr_weighted']:+.3f}")
ax.axhline(pc['corr_unweighted'], color='C0', lw=1, ls='--', label=f"unweighted r {pc['corr_unweighted']:+.3f}")
ax.scatter(pp['n_electrodes'], pp['corr'], s=20)
ax.set(xlabel='electrodes in the participant', ylabel='separate-half LWPC–LWPS r',
       title=f'[{DATA}] the overlap, participant by participant')
ax.legend(frameon=False, fontsize=8)
display(fig); plt.close(fig)

**Exercise E3.** Recombine the per-participant correlations yourself: Fisher
*z* (`np.arctanh`) of each participant's `corr`, weights `max(n − 3, 1)`, the
weighted mean, back to *r* with `np.tanh`. Store it in `my_r`.

<details><summary>Solution</summary>

```python
pp = pc['per_participant']
z = np.arctanh(pp['corr'].clip(-0.999, 0.999))
w = np.maximum(pp['n_electrodes'] - 3, 1)
my_r = np.tanh((w * z).sum() / w.sum())
```
</details>

In [ ]:
my_r = None   # YOUR CODE

In [ ]:
check('E3', my_r, pc['corr_weighted'], hint='arctanh, weights n - 3 (at least 1), weighted mean, tanh.')

**Check yourself, q8.** The weighted participant-level r is significant (p = 0.031) but the unweighted one is not (p = 0.35). What should the paper say?

- (a) The overlap is a property of the electrode population, carried by electrode-rich participants, not shown by every participant
- (b) There is no overlap
- (c) Every participant shows the overlap
- (d) Report only the weighted test

In [ ]:
ask('q8', '?')   # replace ? with your letter

### 4.3 Section 3: shared splits, reliabilities and local similarity

Needs a per-split table scored with one split per participant. It uses
`SHARED_PER_SPLIT` if found, otherwise rescores from the long table (the real
run used 200 splits: several minutes for all lPFC; lower `SHARED_N_SPLITS` to
explore faster).

In [ ]:
if per_split_shared is None and long_df is not None and 'trial' in long_df:
    n = SHARED_N_SPLITS if DATA == 'real' else 24
    print(f'rescoring {long_df["electrode"].nunique()} electrodes with {n} shared splits ...')
    per_split_shared = sfs.compute_sensitivities_per_split(
        long_df[long_df['electrode'].isin(scores['electrode'])], n_splits=n, seed=0,
        contrast_mode='proportion', effect_measure='cohens_d', main_effects=True, shared_split=True)
if per_split_shared is None:
    print('No shared table and no long table with trial ids: section 3 cannot run.')
else:
    print('shared table:', per_split_shared.shape, 'is_shared_split =', sfa.is_shared_split(per_split_shared))

In [ ]:
if per_split_shared is not None:
    rel = sfa._shared_split_reliabilities(per_split, per_split_shared, scores)
    banner('within-participant split-half reliability by split scheme')
    display(rel)

**Inside `local_similarity`, for one participant.** The cell below repeats the
function's first steps by hand: clean each half (responsiveness and the linear
gradient out, centred within participant), rank and scale, then
`C[i, j]` = electrode *i* half A × electrode *j* half B (and the reverse),
averaged over splits. The diagonal is each electrode's reliability. The plot is
`C[i, j]` against distance for every pair of that participant.

In [ ]:
show(sfa.local_similarity, 'coord_cols = list(coord_cols)', 'dev = obs - base')

In [ ]:
from scipy.stats import rankdata

if per_split_shared is not None:
    s_ = scores.drop_duplicates('electrode').set_index('electrode')
    coord_cols = ['mni_x', 'mni_y', 'mni_z']
    have = s_[coord_cols + ['resp']].dropna().index
    ps_, names = sfa._local_score_halves(per_split_shared[per_split_shared['electrode'].isin(have)], scores)
    keys = [k for pair in names.values() for k in pair]
    el_, sj_, grp_, spl_, mats_, _ = sfs._residualised_split_matrices(
        ps_, s_.loc[have, 'resp'], min_elec=3, covariates=s_.loc[have, coord_cols], keys=keys)
    P = s_.loc[el_, coord_cols].to_numpy(float)

    def standardise(Mx):
        Mx = Mx.copy()
        for g in grp_:
            blk = rankdata(Mx[:, g], axis=1)
            Mx[:, g] = blk - blk.mean(axis=1, keepdims=True)
        ms = np.sqrt((Mx ** 2).mean(axis=1, keepdims=True))
        return np.divide(Mx, ms, out=np.zeros_like(Mx), where=ms > 0)

    g = max(grp_, key=len)                         # the participant with most electrodes
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.2), sharey=True)
    for ax, name in zip(axes, ('LWPC', 'LWPS', 'LWPC − LWPS')):
        ka, kb = names[name]
        A, B = standardise(mats_[ka]), standardise(mats_[kb])
        C = 0.5 * (A[:, g].T @ B[:, g] + B[:, g].T @ A[:, g]) / len(spl_)
        iu, ju = np.triu_indices(len(g), 1)
        Dg = np.linalg.norm(P[g][:, None] - P[g][None], axis=-1)
        ax.axhline(0, color='0.8', lw=0.8)
        ax.scatter(Dg[iu, ju], C[iu, ju], s=6, alpha=0.5)
        ax.set(title=f'{name}: reliability (diag) {np.trace(C) / len(g):+.2f}', xlabel='distance (mm)')
    axes[0].set_ylabel('cross-half similarity C[i, j]')
    fig.suptitle(f'[{DATA}] participant {sj_[g[0]]}, {len(g)} electrodes', fontsize=9)
    display(fig); plt.close(fig)

In [ ]:
if per_split_shared is not None:
    loc = sfa.local_similarity(per_split_shared[per_split_shared['electrode'].isin(scores['electrode'])],
                               scores, n_perm=min(N_PERM, 2000), n_boot=N_BOOT)
    banner('local_similarity')
    print('\n'.join(sfa.local_similarity_lines(loc)))
    out = Path(tempfile.mkdtemp()) / 'local_similarity.png'
    sfa.plot_local_similarity(loc, str(out))
    display(Image(str(out)))

How to read it: the single scores are the positive control. On the real data
LWPC and switch show a near-range excess (the analysis can see local
structure), while the balance has neither an excess nor a reliable "same
electrode" row: there is nothing reliable to be patchy (N4 doc §19.8.6).

**Think it through.** A difference score keeps
rel(LWPC) + rel(LWPS) − 2 × (their cross-half correlation) of reliable
variance. Using the shared-split row of the reliability table above, how much
reliable variance should LWPC − LWPS have? Compare with the balance's "same
electrode" row in `loc['table']`.

<details><summary>What you should find</summary>

On the real all-lPFC data: 0.207 + 0.035 − 2 × 0.111 ≈ 0.02, and the balance's
measured reliability is −0.04 (interval up to 0.03). The two adaptations'
reliable variation is mostly shared, so their difference has almost none. That
is why local similarity cannot test "intermixed" here, and why the result reads
"one population rather than two intermixed ones".
</details>

**Check yourself, q9.** In local similarity, what does the diagonal C[i, i] estimate?

- (a) The distance from electrode i to its nearest neighbour
- (b) Electrode i's mean score
- (c) The position-shuffle baseline
- (d) Electrode i's split-half reliability

In [ ]:
ask('q9', '?')   # replace ? with your letter

**Check yourself, q10.** The balance shows no near-range excess. Why can't we conclude it is intermixed?

- (a) Because the p-value is one-sided
- (b) Because the balance has about zero reliable variation, so there is nothing reliable to be patchy or intermixed
- (c) Because the bins are too wide
- (d) Because the positive control failed for LWPC

In [ ]:
ask('q10', '?')   # replace ? with your letter

**Check yourself, q11.** Why does the test flip the signs of whole participants' excesses instead of shuffling pairs?

- (a) Estimation noise is spatially smooth, so pairs are not exchangeable and a pair-level null is too liberal
- (b) Shuffling pairs is too slow
- (c) Pairs have no distances
- (d) To make the test two-sided

In [ ]:
ask('q11', '?')   # replace ? with your letter

### 4.4 Section 4: the combined Figure 5

In [ ]:
fig_dir = Path(tempfile.mkdtemp())
f5 = sfa.figure5_height(scores, str(fig_dir), per_split=per_split, seg_dir=seg_dir,
                        coord_res=coord, participant_res=part, n_perm=N_PERM, n_boot=N_BOOT)
banner('figure5_height')
print('\n'.join(f5['lines']))
display(Image(str(fig_dir / 'fig5_height.png')))

In [ ]:
print('panel c, the tertile centroids:')
display(f5['centroids'])
print('panel d, the balance by tertile (participant means):')
display(f5['balance'])
print('the tertiles on the brain: mean MNI position per tertile and hemisphere (fig5_height_brain_centroids.csv):')
display(f5['brain_centroids'])

The brain version of panel a (added 2026-10-06) needs the recon files and a
display, so it is off by default. On a DCC node, set `DRAW_BRAIN = True` in
cell 2 (start Jupyter under `xvfb-run`, or call `pyvista.start_xvfb()` first).

In [ ]:
if DRAW_BRAIN:
    brain = sfa.plot_height_bands_on_brain(f5['points'].dropna(subset=['band']),
                                           str(fig_dir / 'fig5_height_brain.png'),
                                           centroids=f5['brain_centroids'], edges=f5['edges'],
                                           hemi='split')
    print('fallback (sagittal) used:' if brain['fallback'] else 'rendered:', brain['combined'])
    display(Image(brain['combined']))
else:
    print('DRAW_BRAIN is False: skipped.')

**Check yourself, q12.** In Fig. 5b (LWPC on x, LWPS on y), what does a point above the identity line mean?

- (a) The electrode is dorsal
- (b) Both effects are negative
- (c) The electrode leans LWPS: its LWPS exceeds its LWPC in SD units
- (d) The electrode is an outlier

In [ ]:
ask('q12', '?')   # replace ? with your letter

### 4.5 Section 5: the overlap controls

Each row reruns the overlap test with one more set of covariates removed.
`+ base effects, same half` cleans each half's LWPC and LWPS of that half's
congruency and switch effects.

*Predict:* which row will move *r* the most, and which will barely move it?

In [ ]:
show(sfa.overlap_controls, "run('pre-specified')", "note='descriptive')")

In [ ]:
rt_series = None
if rt is not None:
    rt_series = rt.drop_duplicates('electrode').set_index('electrode')['rt_r']
table, oloso = sfa.overlap_controls(scores, per_split, rt_coupling=rt_series, n_perm=N_PERM)
banner('overlap_controls')
print('\n'.join(sfa.overlap_control_lines(table, oloso)))

**Check yourself, q13.** Partialling the same-half base effects drops r from 0.097 to 0.028. Which reading does this row alone NOT rule out?

- (a) Each adaptation scales with the effect it regulates
- (b) The overlap is real
- (c) RT coupling caused the overlap
- (d) Both adaptations scale with residual signal-to-noise

In [ ]:
ask('q13', '?')   # replace ? with your letter

**Check yourself, q14.** Why are the base effects taken from the **same** half as the adaptation score they clean?

- (a) The other half has no base effects
- (b) So the two sides of the correlation still share no trials
- (c) It makes r larger
- (d) Because of the swap null

In [ ]:
ask('q14', '?')   # replace ? with your letter

## 5 · Play

Everything is in memory: `scores`, `per_split`, `per_split_shared`, `part`
(§4.1), `pc` (§4.2), `loc` (§4.3), `f5` (§4.4), `table` (§4.5). Some things to try:

- Drop the participant with the most weight and refit:
  `sfa.coordinate_slope_by_participant(scores[scores.subject != 'D0xxx'])`.
- The overlap within one hemisphere:
  `sfs.split_resolved_corr(per_split[per_split.electrode.isin(scores[scores.hemi == 'lh'].electrode)], resp)`.
- The sharper base-effect control (an open item): partial LWPC on congruency
  only and LWPS on switch only, with
  `half_covariates={'xA': ('mxA',), 'xB': ('mxB',), 'yA': ('myA',), 'yB': ('myB',)}`.
- Local similarity without removing the gradient:
  `sfa.local_similarity(per_split_shared, scores, remove_gradient=False)`.

In [ ]:
# the sharper base-effect control: each adaptation partialled on its own base effect only
if {'mxA', 'myA'} <= set(per_split.columns):
    own = sfs.split_resolved_corr(per_split, resp, n_perm=N_PERM,
                                  half_covariates={'xA': ('mxA',), 'xB': ('mxB',),
                                                   'yA': ('myA',), 'yB': ('myB',)})
    banner(f"LWPC partialled on congruency, LWPS on switch (same half): r = {own['corr']:+.3f}, "
           f"p = {own['p']:.3g}")